# 01. 기간 그래프와 실제 실험 설정

00의 WORK_DIR/RUN_NAME을 그대로 사용한다. 최초 CSC 생성은 디스크 공간과 시간이 필요하다. 현재 그래프 준비 및 이웃 샘플링은 CPU PyG이고, 모델 학습은 CUDA다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd().resolve()
assert (ROOT / "run.py").exists(), "이 공유 폴더에서 노트북을 열어 주세요."
WORK_DIR = ROOT / "work"
RUN_NAME = "shared_seed42"
def run_cli(*args):
    command = [sys.executable, str(ROOT / "run.py"), "--work-dir", str(WORK_DIR), "--run-name", RUN_NAME, *map(str, args)]
    # Pipe and flush each line so progress is visible in Jupyter.
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as child:
        try:
            for line in child.stdout:
                print(line, end="", flush=True)
            if child.wait() != 0:
                raise RuntimeError(f"실행 실패: {child.returncode}; 위 로그를 확인하세요.")
        except KeyboardInterrupt:
            child.terminate()
            child.wait()
            raise


In [ ]:
import pandas as pd
config = json.loads((ROOT / "configs/shared_500.json").read_text())
display(pd.DataFrame(config.items(), columns=["설정", "값"]))
display(pd.DataFrame(json.loads((ROOT / "results/splits.json").read_text())["windows"]).T)

In [ ]:
RUN_PREPARE = False
if RUN_PREPARE:
    run_cli("prepare", "--config", ROOT / "configs/shared_500.json", "--build-graphs")
else:
    print("입력을 연결한 뒤 RUN_PREPARE=True로 실행하세요.")

## 구현 확인

PE 실험은 ports-only로 논문의 PE(Ports+Ego ID)와 다르다. 전체 코드는 fraudgt_lab/models.py에 있다.

In [ ]:
import inspect
from fraudgt_lab.config import TrainConfig, VARIANTS
from fraudgt_lab.models import make_model, FraudGTBlock
for variant in VARIANTS:
    model = make_model(variant, TrainConfig(**config))
    print(variant, "parameters:", sum(p.numel() for p in model.parameters()), "ego:", model.use_ego, "reverse:", model.reverse)
print(inspect.getsource(FraudGTBlock.forward))